In [ ]:
from pynq import Overlay, allocate, PL
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

PL.reset()
overlay = Overlay('tuto_waveprop(1).bit')


In [ ]:
print('IP blocks :', list(overlay.ip_dict.keys()))


In [ ]:
waveprop = overlay.waveprop_compute_0
dma_recv = overlay.axi_dma_0.recvchannel

CONTROL_REGISTER = 0x0


In [ ]:
width = 100
height = 100

buffer = allocate(shape=(width * height,), dtype=np.int32)

def compute_fpga():
    # Le DMA doit être prêt avant de lancer l'IP
    dma_recv.transfer(buffer)
    waveprop.write(CONTROL_REGISTER, 0x01)
    dma_recv.wait()

    return np.array(buffer, dtype=np.int32).reshape((height, width)).copy()


In [ ]:
%matplotlib inline

# Même principe que wavepropsoft :
# une itération matérielle = une nouvelle image affichée.
for i in range(200):

    start_time = time.time()
    output = compute_fpga()
    processing_time = time.time() - start_time

    clear_output(wait=True)
    plt.imshow(output)
    plt.title(f"Iteration {i+1} time {processing_time}")
    plt.show()


In [ ]:
buffer.freebuffer()
